In [23]:
!pip install -q transformers datasets evaluate scikit-learn

In [24]:
!pip install -q pandas numpy matplotlib

In [25]:
!pip install -q wandb


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.json to test (1).json
Saving train.json to train (1).json
Saving valid.json to valid (1).json


In [27]:
import pandas as pd
import numpy as np
import evaluate

In [28]:
import wandb
wandb.login()

True

In [29]:
data = pd.read_json('train.json')

In [30]:
data['comment_type'].value_counts()

,count
comment_type,
Summary,8398


In [31]:
data['label'].value_counts()

,count
label,
1,4199
0,4199


In [32]:
val = pd.read_json('valid.json')[['new_comment_raw', 'new_code_raw', 'label']]
test = pd.read_json('test.json')[['new_comment_raw', 'new_code_raw', 'label']]
val.head()

,new_comment_raw,new_code_raw,label
0,Return SQL selector query for getting tasks wi...,public static QueryTemplate queryTempl...,1
1,Return period of ghost connections cleanup tas...,public int getGhostConnsCleanupPeriod() {\...,0
2,Allocates an initialized and initially unlocke...,\tpublic static SpinLock allocateSpinLock() {\...,1
3,Rebalance a frame for load balancing,private static Frame reBalance(final Frame f...,0
4,Sets the host of the proxy.,public Proxy setHost( String host )\n {...,1


In [33]:
train = data[['new_comment_raw', 'new_code_raw', 'label']]

"""Why should we combine code and comment together for GraphCodeBERT??

* GraphCodeBERT is not designed to encode two independent inputs in isolation.
  Like CodeBERT, it expects a single sequence.

* GraphCodeBERT was pretrained jointly on code and natural language (e.g., docstrings),
  using the format:

    [CLS] code tokens [SEP] comment tokens [SEP]

* By concatenating them into one sequence, GraphCodeBERT can attend across both
  the code and the comment tokens, leveraging self-attention to capture whether
  the natural language description aligns with the source code.

* Additionally, GraphCodeBERT incorporates data flow edges during pretraining,
  which strengthens its ability to model relationships inside the code while still
  considering the natural language description.
"""


In [34]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [35]:
from datasets import Dataset
from datasets import DatasetDict

train = Dataset.from_pandas(train)
test = Dataset.from_pandas(test)
val = Dataset.from_pandas(val)

dataset = DatasetDict({
    "train": train,
    "validation": val,
    "test": test
})

In [36]:
def preprocess_function(examples):
    return tokenizer(
        examples["new_code_raw"],
        examples["new_comment_raw"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

In [37]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/8398 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [38]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['new_comment_raw', 'new_code_raw', 'label', 'input_ids', 'attention_mask'],
        num_rows: 8398
    })
    validation: Dataset({
        features: ['new_comment_raw', 'new_code_raw', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1034
    })
    test: Dataset({
        features: ['new_comment_raw', 'new_code_raw', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [39]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [40]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall": recall.compute(predictions=preds, references=labels)["recall"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"]
    }


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("microsoft/graphcodebert-base", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="wandb"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-118606857.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.515600,0.297107,0.866538,0.997375,0.735010,0.846325
2,0.376800,0.322449,0.849130,0.866126,0.825919,0.845545
3,0.241900,0.447501,0.828820,0.798246,0.880077,0.837167
4,0.136700,0.618603,0.845261,0.852071,0.835590,0.843750
5,0.063500,0.770441,0.845261,0.857715,0.827853,0.842520


TrainOutput(global_step=2625, training_loss=0.25650659542992, metrics={'train_runtime': 4316.0168, 'train_samples_per_second': 9.729, 'train_steps_per_second': 0.608, 'total_flos': 1.10480332145664e+16, 'train_loss': 0.25650659542992, 'epoch': 5.0})

In [ ]:
import wandb
test_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
print(test_results)

wandb.log(test_results)

print("Metrics logged to run:", wandb.run.name, "(id:", wandb.run.id, ")")
wandb.finish()

{'eval_loss': 0.328652948141098, 'eval_accuracy': 0.8480300187617261, 'eval_precision': 0.9946666666666667, 'eval_recall': 0.699812382739212, 'eval_f1': 0.8215859030837004, 'eval_runtime': 30.6361, 'eval_samples_per_second': 34.796, 'eval_steps_per_second': 2.187, 'epoch': 5.0}
Metrics logged to run: lemon-sea-10 (id: n9ngmya5 )


epoch,▁
eval/accuracy,█▅▁▄▄▅
eval/f1,██▅▇▇▁
eval/loss,▁▁▃▆█▁
eval/precision,█▃▁▃▃█
eval/recall,▂▆█▆▆▁
eval/runtime,▁▁▁▁▁█
eval/samples_per_second,▇▇▇█▇▁
eval/steps_per_second,▇▇▇█▇▁
eval_accuracy,▁
+12,...
